# Campus SVI — analysis

Reads the acquisition deliverables and produces tables and publication figures.
**Fetches nothing** — run `01_acquisition.ipynb` first.

| Reads | Writes |
|---|---|
| `data/cells/{campus}_cells.gpkg` | `data/analysis/tables/*.csv` |
| `data/points/{campus}_points.gpkg` | `data/analysis/figures/*.pdf` and `.png` |
| `boundaries/*.gpkg` | `data/reference/campus_registry.csv` |

The grid is **20 m**. Coverage ratios are lower than at a coarser cell by construction — a
smaller cell is harder to intersect — so these numbers are not comparable with results computed
at another cell size. State the size wherever a ratio is reported.

---
## 1 · Setup


In [ ]:
#@title Install
!pip install -q geopandas pyogrio matplotlib
print('ok')


In [ ]:
#@title Mount Drive and load the repo
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/aditpradana36/campus-svi-availability.git'  #@param {type:'string'}
PROJECT_ROOT = '/content/drive/MyDrive/campus-svi-availability'  #@param {type:'string'}

import os, sys
REPO_DIR = '/content/campus-svi-acquisition'
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from campus_svi import config, registry
config.set_root(PROJECT_ROOT)

from campus_svi.analysis import metrics, figures, maps, style as st
import pandas as pd, numpy as np, matplotlib.pyplot as plt

CAMPUSES = metrics.available()
print(f'{len(CAMPUSES)} campuses with cell data')
print(', '.join(registry.display_names(CAMPUSES)))


---
## 2 · Campus registry

Identity and location for every campus. Centroid, area, perimeter and UTM zone are derived from
your boundary files; display name, city and province come from the lookup in
`campus_svi/registry.py` and are **hand-entered**.

Multi-site universities keep the institution's own designation — UNESA Campus 1 and 2, UNAIR
Campus B and C — rather than place names, since those are the official labels.

Every row is written with `verified = False`. Check the city and province column once, then set
it True. Three abbreviations differ from their slug and are worth a second look: `unbraw` is UB,
`unlam` is ULM, `unud_jimbaran` is UDAYANA.


In [ ]:
#@title Build the registry CSV
reg_path = registry.write(CAMPUSES)
reg = registry.load()

display(reg[['campus_id','display_name','city','province','island',
             'centroid_lat','centroid_lon','area_km2','n_cells']])


---
## 3 · Tables

Numbers first. Every value in a figure is also written as a CSV, so anything surprising in a
plot can be checked against the table rather than read off the pixels.


In [ ]:
#@title Build every table
paths = metrics.write_tables(CAMPUSES)

cov = metrics.coverage_table(CAMPUSES)
display(cov.sort_values('either_coverage', ascending=False)
        [['display_name','n_cells','mly_coverage','ggl_coverage','either_coverage']]
        .round(3).head(12))


In [ ]:
#@title Headline numbers
print(f"campuses         : {len(cov)}")
print(f"grid cells (20 m): {cov['n_cells'].sum():,}")
print(f"Mapillary images : {cov['mly_images'].sum():,}")
print(f"Google panoramas : {cov['ggl_panoramas'].sum():,}")
print()
print(f"mean Mapillary coverage : {cov['mly_coverage'].mean():.1%}")
print(f"mean Google coverage    : {cov['ggl_coverage'].mean():.1%}")
print(f"mean either-source      : {cov['either_coverage'].mean():.1%}")
print()
for k in ('both','mapillary_only','google_only','neither'):
    print(f"  {k:<16} {cov[f'prop_{k}'].mean():.1%}")


---
## 4 · Coverage and agreement


In [ ]:
#@title Figure 1 — coverage by campus
fig, ax = figures.fig_coverage(CAMPUSES)
plt.show()


### On the maps

Each panel is **fitted to its own campus boundary**, so every campus fills its frame regardless
of size. That keeps internal structure legible on a small campus and a large one alike.

Because scale therefore differs between panels, **each panel carries its own scale bar**, placed
below the frame — panels are filled edge to edge, so there is no reliable empty corner inside.
A single shared bar would be false here.

What panel size no longer encodes is campus extent. The bars carry it; set `show_area=True` to
print each campus's area beside its name as well.

At 20 m a campus can carry thousands of cells, so the cell layer is rasterised per panel at
600 dpi while boundaries, text and bars stay vector — identical in print, but a PDF that opens.

**Panel order is canonical across every figure**, beginning with IPB, so a campus keeps its
position from one figure to the next. Sorting each figure by its own metric would make each one
internally tidy but mutually incomparable, which is the more expensive mistake across a set this
size.


In [ ]:
#@title Figure 2 — agreement maps
NCOLS = 8  #@param {type:'integer'}
SHOW_AREA = False  #@param {type:'boolean'}

fig, axes = figures.fig_agreement_maps(CAMPUSES, ncols=NCOLS, show_area=SHOW_AREA)
plt.show()


In [ ]:
#@title Figure 2b — agreement composition
fig, ax = figures.fig_agreement_composition(CAMPUSES)
plt.show()


---
## 5 · Enclosure — the depth decay test

Distance from each cell centroid to the campus edge, normalised by that campus's own maximum so
0 is the perimeter and 1 the deepest interior point. Road-free: it needs only the boundary.

The **slope** of coverage against normalised depth is the openness index. Steeply negative means
coverage collapses inward — a more enclosed campus. This does double duty as both the test of
the enclosure hypothesis and the openness measure used to compare campuses.


In [ ]:
#@title Figure 3 — decay curves
HIGHLIGHT = ''  #@param {type:'string'}
hl = [c.strip() for c in HIGHLIGHT.split(',') if c.strip()]

fig, ax = figures.fig_decay(CAMPUSES, highlight=hl)
plt.show()

sl = metrics.decay_slope(CAMPUSES).sort_values('openness_slope')
sl['name'] = registry.display_names(sl['campus_id'])
display(sl[['name','openness_slope','edge_coverage','core_coverage','r2']].round(3).head(10))


---
## 6 · Temporal

**Figure 4** — distinct Google capture years per cell.

**Figure 4b** — Mapillary minus Google capture years where both sources cover the cell.
Diverging ramp centred on zero, a real midpoint here. Cells covered by one source are blank,
since the comparison is undefined there.

**Figure 5** — annual volume by source, the monthly Mapillary series, and burstiness against
contributor concentration.


In [ ]:
#@title Figures 4 — temporal depth, each source
figures.fig_temporal_depth(CAMPUSES, 'google', ncols=NCOLS, show_area=SHOW_AREA)
plt.show()
figures.fig_temporal_depth(CAMPUSES, 'mapillary', ncols=NCOLS, show_area=SHOW_AREA)
plt.show()
figures.fig_depth_diff(CAMPUSES, ncols=NCOLS, show_area=SHOW_AREA)
plt.show()


In [ ]:
#@title Figure 5 — temporal signature
fig, axes = figures.fig_temporal_signature(CAMPUSES)
plt.show()

sig = metrics.temporal_signature(CAMPUSES)
sig['name'] = registry.display_names(sig['campus_id'])
display(sig[['name','n_months_active','cv_monthly','top_creator_share','n_sequences']]
        .round(3).head(12))


---
## 8 · Robustness

**MAUP** at 20 / 50 / 100 m, spanning the working resolution. Ratios rise with cell size by
construction, so what matters is whether the campus ranking and the between-source gap survive.
Cheap here: the point deliverable is resolution-independent, so re-gridding needs no refetching.

**Moran's I** because adjacent cells are not independent — without it, any cell-level
significance claim is unsupported. Computed on the lattice from row/column adjacency with a
permutation test.


In [ ]:
#@title Figure S1 and the MAUP table
SIZES = [20, 50, 100]  #@param

fig, axes = figures.fig_maup(CAMPUSES, sizes=tuple(SIZES))
plt.show()

maup = metrics.maup_table(CAMPUSES, sizes=tuple(SIZES))
maup.to_csv(config.DATA_DIR / 'analysis' / 'tables' / 'maup_wide.csv', index=False)
display(maup.round(3))

#@markdown Coverage rises with cell size by construction, so the absolute
#@markdown numbers are not the finding. What matters is whether the campus
#@markdown *ranking* survives — the correlations below.
long = metrics.maup_profile(CAMPUSES, sizes=tuple(SIZES))
piv = long.pivot_table(index='campus_id', columns='cell_size_m',
                       values='either_coverage')
print('\nrank correlation of campus coverage between cell sizes:')
print(piv.corr(method='spearman').round(3))


In [ ]:
#@title Figure S2 — Moran's I
fig, ax = figures.fig_autocorrelation(CAMPUSES)
plt.show()

ac = metrics.autocorrelation_table(CAMPUSES)
sig_n = len(ac[(ac['column']=='either_coverage') & (ac['p_sim']<0.05)])
print(f"{sig_n}/{ac['campus_id'].nunique()} campuses show significant clustering")


---
## 10 · Record counts

Literal counts per cell, each source on its own map, in **two versions**.

The **log** version shows pattern: counts are heavily right-skewed — a few cells on a
well-travelled path hold dozens of images while most hold one or two — and a linear ramp
flattens that tail to the bottom colour. The **raw** version shows magnitude without
transformation, which is the honest picture of how much imagery a cell holds even though
most cells end up near the floor.

Cells with zero records are drawn as *no data* on the log version, since zero has no position
on a log axis.

Neither map can answer *how much imagery does this campus have in total* — a per-cell ramp
compresses exactly the between-campus differences that question is about. The bar charts
below give that directly, on a linear axis with the values printed.


In [ ]:
#@title Figures 7 — counts per cell, and campus totals
#@markdown Log and raw versions of each map. The log version shows
#@markdown *pattern* — where records concentrate — while the raw one
#@markdown shows *magnitude* honestly, at the cost of flattening most
#@markdown cells. Publish whichever answers your sentence; keep both
#@markdown while you decide.
for src_ in ('mapillary', 'google'):
    figures.fig_count_maps(CAMPUSES, src_, ncols=NCOLS,
                           show_area=SHOW_AREA, log=True)
    plt.show()
    figures.fig_count_maps(CAMPUSES, src_, ncols=NCOLS,
                           show_area=SHOW_AREA, log=False)
    plt.show()


In [ ]:
#@title Figure 7c — total records per campus
#@markdown Google left, Mapillary right, on independent scales: the two
#@markdown sources differ in volume by an order of magnitude, so a shared
#@markdown scale would flatten one side. Bar lengths are therefore not
#@markdown comparable across the centre line — campuses are.
figures.fig_count_bars(CAMPUSES)
plt.show()


---
## 11 · Capture volume per year, per campus

One small bar chart per campus, each source separately.

Axes are **not shared** by default: campuses differ in volume by orders of magnitude, and a
common axis flattens the small ones to a flat line. Each panel is therefore about *timing* —
when capture happened — with the peak-year count annotated so magnitude is still recoverable.
Set `share_y=True` if you want volume comparable across panels instead.


In [ ]:
#@title Figures 8 — annual bars
figures.fig_annual_bars(CAMPUSES, 'mapillary', ncols=NCOLS)
plt.show()
figures.fig_annual_bars(CAMPUSES, 'google', ncols=NCOLS)
plt.show()


---
## 12 · Local Moran's I per campus

Global Moran's I (section 8) says whether a campus is clustered. It cannot say **where**.
The local statistic decomposes it per cell, so a coverage gap in the campus core shows up as a
contiguous Low-Low cluster rather than disappearing into a single number.

Two figures per source:

| Figure | Shows |
|---|---|
| `fig9_*_local_moran` | the value of I per cell — strength of local association |
| `fig9b_*_quadrants` | the cluster type — High-High, Low-Low, and the outliers |

For this project the **Low-Low clusters are the finding**: contiguous interior areas no source
reaches. Cells failing the permutation test are drawn near-white and mean *no signal*, not a
weak one — do not read them as low values.

Weights are row-standardised lattice adjacency from the row/column indices, so adjacency is
exact and no spatial-weights library is needed.


In [ ]:
#@title Figures 9 — Mapillary LISA
figures.fig_local_moran(CAMPUSES, 'mapillary', ncols=NCOLS, show_area=SHOW_AREA)
plt.show()
figures.fig_moran_quadrants(CAMPUSES, 'mapillary', ncols=NCOLS, show_area=SHOW_AREA)
plt.show()


In [ ]:
#@title Figures 9 — Google LISA
figures.fig_local_moran(CAMPUSES, 'google', ncols=NCOLS, show_area=SHOW_AREA)
plt.show()
figures.fig_moran_quadrants(CAMPUSES, 'google', ncols=NCOLS, show_area=SHOW_AREA)
plt.show()


In [ ]:
#@title LISA summary table
lisa = metrics.local_morans_summary(CAMPUSES, 'mly_count')
lisa['name'] = registry.display_names(lisa['campus_id'])
display(lisa[['name','n_cells','prop_HH','prop_LL','prop_LH','prop_HL',
              'prop_significant']].round(3).head(12))


---
## 13 · Contributors

Two stacked panels on a shared campus axis. (a) counts contributors per campus; (b) places one
dot per contributor at the number of images they uploaded, so a campus with three contributors
shows three dots at three different heights.

The pairing is the point. Panel (a) alone would say a campus is well-contributed; (b) shows
whether that is twenty people sharing the work or one person carrying it while nineteen added a
handful each. **Two campuses with identical coverage can be entirely different phenomena
underneath** — which is why this sits beside the coverage figures rather than in an appendix.

Colours are institutional and shared with the study-area map, so a campus is recognisable
across the figure set.


In [ ]:
#@title Figure 10 — contributors per campus
DOT_SIZE = 16  #@param {type:'number'}
DOT_EDGE = 0.35  #@param {type:'number'}
DOT_EDGE_COLOR = 'white'  #@param ['white', 'black', 'none']
GINI_FONTSIZE = 7.5  #@param {type:'number'}

#@markdown Dot size depends on how many contributors your campuses
#@markdown have — tune once you see real data. `GINI_FONTSIZE` sets the
#@markdown numbers on panel (c), which sits in a short panel and needs
#@markdown larger type than the panels above to stay readable.

figures.fig_contributors(CAMPUSES, dot_size=DOT_SIZE, dot_edge=DOT_EDGE,
                         dot_edge_color=DOT_EDGE_COLOR,
                         gini_fontsize=GINI_FONTSIZE)
plt.show()

con = metrics.contributor_summary(CAMPUSES)
con['name'] = registry.display_names(con['campus_id'])
display(con[['name','n_contributors','n_images','top1_share','gini',
             'n_for_90pct']].sort_values('gini', ascending=False).round(3))


---
## 9 · Build everything


In [ ]:
#@title All tables and figures
registry.write(CAMPUSES)
metrics.write_tables(CAMPUSES)
figures.build_all(CAMPUSES, ncols=NCOLS, show_area=SHOW_AREA)
print(f"\noutputs under {config.DATA_DIR / 'analysis'}")


---
## Notes

**Cell size.** 20 m. Report it wherever a coverage ratio appears; the numbers do not transfer
across cell sizes.

**Scope of inference.** Cell-level statistics within a campus rest on thousands of cells at 20 m
and are fine. Across campuses, n = 40 — descriptive typology, not regression.

**Date precision is not uniform.** Official Google coverage is month-level, third-party uploads
can carry a day, Mapillary timestamps are millisecond-precise. Bin to year, or at finest month.

**Restyling.** Colours in `analysis/style.py`, house style in `analysis/paperstyle.py`, names in
`registry.py`. Change them there and every figure follows.


---
## Study area maps

The Indonesia overview and the per-campus imagery panels live in **`03_study_area.ipynb`**.
They need only the boundary files, not acquired data, so they can be produced at any point.
